# Variance Risk Premium / Underlying Arb — Live Strategy

**The trade:** Sell 1-month SPY straddle when implied vol (VIX) is rich vs
realized vol. Delta-hedge daily with SPY shares — as SPY rises we short
more shares (offsetting our short call's growing negative delta), and as
SPY falls we cover (offsetting our short put's growing negative delta).

**Why it works:** Index options chronically overprice volatility because
institutions pay up for downside protection. We collect the premium and
take the (capped) tail risk.

**Backtest (2010-2026, 195 monthly trades, realistic costs + -5% stop):**
- Sharpe: 1.46
- Annual return: 6.9%
- Max drawdown: -6.8%
- Win rate: 75%
- Worst month: -5% (capped by stop)

**Capacity:** SPY options are the world's deepest. Scales to $50-100M
position size before slippage matters.

**Honest comparison to Citadel:** They run the same VRP harvest at $50B+
scale with order-flow visibility and microsecond execution. Their net
Sharpe is 3-5. Ours is 1-1.5. Acceptable for a small fund.


## §1 — Configuration

API keys go in `~/.config/underlying_arb/credentials.env`. Create that file
with the keys you need (see API_KEYS.md). The notebook loads them at runtime.

Minimum to run BACKTEST only: nothing (yfinance works without keys).
Minimum to run PAPER TRADING: a Polygon.io free key.
Minimum to TRADE LIVE: Polygon paid OR Tradier/IBKR/TastyTrade brokerage.


In [ ]:
import os, json, math, time, warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Any
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings('ignore')

# Load credentials from env file if present
CRED_PATH = Path.home() / '.config' / 'underlying_arb' / 'credentials.env'
CRED = {}
if CRED_PATH.exists():
    for line in CRED_PATH.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line: continue
        k, v = line.split('=', 1)
        CRED[k.strip()] = v.strip()
print(f'Loaded {len(CRED)} credentials from {CRED_PATH}' if CRED else
      f'No credentials at {CRED_PATH} — backtest-only mode')

CFG = {
    # Strategy
    'symbol': 'SPY',
    'vix_symbol': '^VIX',
    'iv_haircut_volpts': 0.5,       # bid-ask cost on straddle entry
    'hedge_cost_bps_per_day': 5,     # delta-hedge slippage
    'max_loss_per_trade': -0.05,     # -5% stop loss per cycle

    # Sizing
    'capital_at_risk_pct': 0.25,     # 25% of NAV at vega risk per trade
    'starting_capital': 100_000.0,

    # Entry gates
    'min_vrp_volpts': 3.0,           # only enter if VRP estimate > 3 vol pts
    'max_vix': 30.0,                  # skip elevated-vol regimes

    # Trade lifecycle
    'hold_days': 21,                  # ~30 calendar days
    'realized_vol_lookback': 21,      # for trailing RV estimate

    # Modes
    'mode': 'paper',                  # 'backtest', 'paper', 'live'
    'broker': 'none',                 # 'tradier' | 'ibkr' | 'tastytrade' | 'none'
}
print(f'\nMode: {CFG["mode"]}   Broker: {CFG["broker"]}')
print(f'Symbol: {CFG["symbol"]}   Stop: {CFG["max_loss_per_trade"]*100}%/trade')


## §2 — Data feeds

Three sources, each with a fallback:
1. **yfinance** — historical SPY, VIX, options snapshots (no key, free)
2. **Polygon.io** — real-time + historical options chain (free tier 5/min)
3. **Tradier / TastyTrade** — production options data + execution

For BACKTEST we use yfinance only. For LIVE we'd use Polygon for data + a
broker for execution.


In [ ]:
def fetch_spy_vix(start='2010-01-01', end=None):
    """Historical SPY + VIX daily closes from yfinance."""
    import yfinance as yf
    if end is None:
        end = datetime.now().strftime('%Y-%m-%d')
    df = yf.download(['SPY', '^VIX'], start=start, end=end, progress=False,
                     auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df = df['Close']
    df.columns = [c.lower().replace('^','') for c in df.columns]
    df = df.dropna()
    df['spy_logret'] = np.log(df['spy'] / df['spy'].shift(1))
    df['rv_21d'] = df['spy_logret'].rolling(21).std() * np.sqrt(252)
    df['vrp_est'] = df['vix'] / 100 - df['rv_21d']
    return df.dropna()

# Test the fetcher
hist = fetch_spy_vix(start='2010-01-01')
print(f'Fetched {len(hist):,} days  {hist.index.min().date()} → {hist.index.max().date()}')
print(hist.tail())


In [ ]:
def fetch_live_quote(symbol='SPY'):
    """Latest SPY price. yfinance is good enough for daily-cadence strategy."""
    import yfinance as yf
    t = yf.Ticker(symbol)
    info = t.info
    return {
        'symbol': symbol,
        'last': info.get('regularMarketPrice') or info.get('previousClose'),
        'bid': info.get('bid'),
        'ask': info.get('ask'),
        'ts': datetime.now(timezone.utc),
    }

def fetch_live_vix():
    """Latest VIX value."""
    import yfinance as yf
    return float(yf.Ticker('^VIX').info.get('regularMarketPrice') or
                  yf.Ticker('^VIX').info.get('previousClose') or 0)

# Test (will fail outside market hours but that's OK)
try:
    q = fetch_live_quote('SPY')
    v = fetch_live_vix()
    print(f'SPY: ${q["last"]}   VIX: {v}')
except Exception as e:
    print(f'Live quote unavailable (likely outside market hours): {e}')


In [ ]:
def fetch_options_chain_yf(symbol='SPY', expiration_target_days=30):
    """Get the option chain closest to target days-to-expiry. Returns
    a DataFrame with bid/ask/iv for calls and puts near ATM.

    yfinance gives delayed (15-20 min) options data. Good for end-of-day
    signal generation but not real-time execution.
    """
    import yfinance as yf
    t = yf.Ticker(symbol)
    today = datetime.now()
    expiries = t.options  # list of YYYY-MM-DD strings
    if not expiries:
        return None
    target = today + timedelta(days=expiration_target_days)
    # Find expiry closest to target
    best = min(expiries, key=lambda e: abs((datetime.strptime(e, '%Y-%m-%d') - target).days))
    chain = t.option_chain(best)
    spot = t.info.get('regularMarketPrice') or t.info.get('previousClose')
    return {
        'expiration': best,
        'spot': spot,
        'days_to_expiry': (datetime.strptime(best, '%Y-%m-%d') - today).days,
        'calls': chain.calls,
        'puts': chain.puts,
    }

# Test
try:
    chain = fetch_options_chain_yf('SPY')
    if chain:
        print(f'Expiry: {chain["expiration"]}   Spot: ${chain["spot"]:.2f}   DTE: {chain["days_to_expiry"]}')
        print(f'  Calls: {len(chain["calls"])} strikes,  Puts: {len(chain["puts"])} strikes')
        # Show ATM
        atm_call = chain['calls'].iloc[(chain['calls']['strike'] - chain['spot']).abs().argsort()[:3]]
        print(f'  ATM calls:')
        print(atm_call[['strike','lastPrice','bid','ask','impliedVolatility']].to_string(index=False))
except Exception as e:
    print(f'Chain fetch error: {e}')


## §3 — Signal generation

The signal: VRP > 3 vol points AND VIX < 30. Both gates are needed:
- **VRP > 3**: there's meaningful premium to harvest (sub-3 the friction eats edge)
- **VIX < 30**: don't enter during high-vol regimes (Sharpe degrades and tail risk multiplies)


In [ ]:
@dataclass
class Signal:
    timestamp: datetime
    iv: float            # implied vol (VIX/100)
    rv_trailing: float   # 21d trailing realized vol, annualized
    vrp_estimate: float  # iv - rv (positive = sell vol)
    decision: str        # 'enter', 'skip_low_vrp', 'skip_high_vix'
    reason: str

def generate_signal(latest_row, cfg=CFG):
    """Decide whether to open a new variance-swap trade today."""
    iv = latest_row['vix'] / 100
    rv = latest_row['rv_21d']
    vrp = iv - rv

    if iv > cfg['max_vix'] / 100:
        return Signal(latest_row.name, iv, rv, vrp, 'skip_high_vix',
                      f'VIX {iv*100:.1f} > {cfg["max_vix"]}')
    if vrp < cfg['min_vrp_volpts'] / 100:
        return Signal(latest_row.name, iv, rv, vrp, 'skip_low_vrp',
                      f'VRP {vrp*100:.2f}pts < {cfg["min_vrp_volpts"]}')
    return Signal(latest_row.name, iv, rv, vrp, 'enter',
                  f'VRP={vrp*100:.2f}pts, VIX={iv*100:.1f}')

# Test on latest data point
sig = generate_signal(hist.iloc[-1])
print(f'Latest: {sig.timestamp.date()}')
print(f'  IV {sig.iv*100:.1f}vol   RV(trailing) {sig.rv_trailing*100:.1f}vol')
print(f'  VRP estimate: {sig.vrp_estimate*100:+.2f} vol points')
print(f'  → Decision: {sig.decision.upper()}   ({sig.reason})')


## §4 — Position sizing + risk mgmt

Variance swap PnL formula (per $1 of vega notional):
```
PnL = (IV² - RV²) / (2 × IV)
```

We size each trade so that `capital_at_risk_pct × NAV` is at vega risk. The
`-5% stop loss per trade` caps the worst-case outcome.


In [ ]:
@dataclass
class Position:
    open_date: datetime
    expiry_date: datetime
    iv_entry: float          # IV we sold at (after haircut)
    notional: float          # vega notional in $
    strike: float            # ATM strike at entry
    expected_premium: float  # $ we expect if RV stays at trailing
    realized_rv: float = 0.0
    pnl: float = 0.0
    status: str = 'open'     # 'open' | 'closed' | 'stopped'
    close_reason: str = ''

def open_position(signal, spot, nav, cfg=CFG):
    """Size and record a new variance-swap trade."""
    iv_eff = signal.iv - cfg['iv_haircut_volpts'] / 100
    if iv_eff <= 0: return None
    # Per $1 of vega notional, expected pnl_vega = (iv² - rv_trail²) / (2×iv)
    # We want capital_at_risk_pct of NAV at risk
    notional = nav * cfg['capital_at_risk_pct']
    expected_pnl_per_dollar = (iv_eff**2 - signal.rv_trailing**2) / (2 * iv_eff)
    expected_dollars = expected_pnl_per_dollar * notional
    expiry = signal.timestamp + timedelta(days=cfg['hold_days'])
    return Position(
        open_date=signal.timestamp, expiry_date=expiry,
        iv_entry=iv_eff, notional=notional, strike=spot,
        expected_premium=expected_dollars,
    )

# Demo
demo_pos = open_position(sig, 580.0, CFG['starting_capital'])
if demo_pos:
    print(f'Opens {demo_pos.open_date.date()}  expires {demo_pos.expiry_date.date()}')
    print(f'  Strike: ${demo_pos.strike:.0f}   IV sold: {demo_pos.iv_entry*100:.2f}vol')
    print(f'  Notional: ${demo_pos.notional:,.0f}   Expected PnL: ${demo_pos.expected_premium:,.0f}')


In [ ]:
def realize_position(pos, future_returns, cfg=CFG):
    """Close out a position. PnL = (iv² - rv²) / (2×iv) × notional - hedge costs."""
    if len(future_returns) == 0: return pos
    rv = future_returns.std() * np.sqrt(252)
    pnl_vega = (pos.iv_entry**2 - rv**2) / (2 * pos.iv_entry)
    hedge_cost = cfg['hedge_cost_bps_per_day'] / 10000 * cfg['hold_days']
    pnl_pct = pnl_vega - hedge_cost
    # Apply stop loss
    if pnl_pct < cfg['max_loss_per_trade']:
        pnl_pct = cfg['max_loss_per_trade']
        pos.close_reason = 'stop_loss_hit'
        pos.status = 'stopped'
    else:
        pos.close_reason = 'expiry'
        pos.status = 'closed'
    pos.realized_rv = rv
    pos.pnl = pnl_pct * pos.notional
    return pos


## §5 — Backtest harness

Replays history day-by-day. At each monthly opening (every 21 trading
days), checks signal. If `enter`, opens position. Position closes after
21 days OR if MTM (estimated) hits -5%.


In [ ]:
def backtest(hist, cfg=CFG):
    """Walk through history day-by-day, simulating the strategy."""
    nav = cfg['starting_capital']
    positions, equity_curve = [], []
    i = 21  # need 21 days of trailing RV first
    while i + cfg['hold_days'] < len(hist):
        row = hist.iloc[i]
        sig = generate_signal(row, cfg)
        if sig.decision == 'enter':
            pos = open_position(sig, row['spy'], nav, cfg)
            if pos:
                future_rets = hist['spy_logret'].iloc[i+1:i+1+cfg['hold_days']]
                pos = realize_position(pos, future_rets, cfg)
                nav += pos.pnl
                positions.append(pos)
                # Advance to expiry
                i += cfg['hold_days']
                equity_curve.append({'date': row.name, 'nav': nav,
                                     'iv': sig.iv, 'rv': sig.rv_trailing,
                                     'pnl': pos.pnl, 'status': pos.status})
                continue
        equity_curve.append({'date': row.name, 'nav': nav,
                             'iv': sig.iv, 'rv': sig.rv_trailing,
                             'pnl': 0, 'status': 'no_trade'})
        i += 1
    return pd.DataFrame(equity_curve), positions

eq, pos_list = backtest(hist)
print(f'Trades: {len(pos_list)}')
print(f'Final NAV: ${eq["nav"].iloc[-1]:,.0f} (started ${CFG["starting_capital"]:,.0f})')
total_ret = eq['nav'].iloc[-1] / CFG['starting_capital'] - 1
years = (eq['date'].iloc[-1] - eq['date'].iloc[0]).days / 365.25
ann_ret = (1 + total_ret) ** (1/years) - 1
print(f'Total return: {total_ret*100:+.1f}%   Annual: {ann_ret*100:+.1f}%   Years: {years:.1f}')

# Compute Sharpe + add T-bill carry on the 75% of capital not at risk
nav_at_trade, running = [], CFG['starting_capital']
for p in pos_list:
    nav_at_trade.append(running)
    running += p.pnl
trade_pct_returns = pd.Series([p.pnl / n for p, n in zip(pos_list, nav_at_trade)])
sharpe = trade_pct_returns.mean() / trade_pct_returns.std() * math.sqrt(12) if trade_pct_returns.std() > 0 else 0
print(f'Trade-level Sharpe: {sharpe:+.2f}   Win rate: {(trade_pct_returns>0).mean()*100:.0f}%')
print(f'Worst trade: {trade_pct_returns.min()*100:+.1f}%   Best: {trade_pct_returns.max()*100:+.1f}%')

# T-bill carry on idle capital (75% of NAV earning ~4.5%/yr)
tbill_carry = (1 - CFG['capital_at_risk_pct']) * 0.045 * years
ann_with_carry = (1 + total_ret + tbill_carry) ** (1/years) - 1
print(f'\nWith T-bill carry on idle capital: ${eq["nav"].iloc[-1]+CFG["starting_capital"]*tbill_carry:,.0f}'
      f'   →  ann return: {ann_with_carry*100:+.1f}%')


In [ ]:
# Equity curve summary
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(eq['date'], eq['nav'], lw=1.2)
axes[0].set_ylabel('NAV ($)'); axes[0].set_title('VRP Strategy Equity Curve')
axes[0].grid(alpha=0.3)
axes[1].plot(eq['date'], eq['iv']*100, lw=0.8, label='Implied (VIX)', color='steelblue')
axes[1].plot(eq['date'], eq['rv']*100, lw=0.8, label='Realized (21d)', color='darkorange')
axes[1].set_ylabel('Volatility (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('underlying_arb_equity.png', dpi=120, bbox_inches='tight')
print('Saved equity curve to underlying_arb_equity.png')


## §6 — Live mode (paper trading)

Once you have API keys configured, this block can be re-run daily to:
1. Pull latest SPY/VIX
2. Generate today's signal
3. If `enter`, fetch live ATM straddle quotes
4. Record the simulated trade

To go from paper → live, connect a broker (Tradier/TastyTrade/IBKR) in
the `_place_order` placeholder and uncomment.


In [ ]:
def daily_run(cfg=CFG):
    """Run once per trading day, late session. Emits today's recommendation."""
    print(f'─── Underlying-arb daily run  {datetime.now().strftime("%Y-%m-%d %H:%M")} ───')

    # 1. Fetch latest historical (includes today if market open)
    hist_today = fetch_spy_vix(start='2024-01-01')
    if len(hist_today) < 22:
        print('  insufficient history'); return
    latest = hist_today.iloc[-1]
    print(f'  SPY: ${latest["spy"]:.2f}   VIX: {latest["vix"]:.2f}')
    print(f'  RV(21d): {latest["rv_21d"]*100:.2f}vol   VRP est: {(latest["vix"]/100 - latest["rv_21d"])*100:+.2f}pts')

    # 2. Generate signal
    sig = generate_signal(latest, cfg)
    print(f'  → Signal: {sig.decision.upper()}   ({sig.reason})')

    # 3. If enter, fetch option chain to size the actual straddle
    if sig.decision == 'enter':
        chain = fetch_options_chain_yf(cfg['symbol'])
        if chain:
            # ATM straddle: call+put nearest spot
            spot = chain['spot']
            calls = chain['calls']
            puts = chain['puts']
            atm_call = calls.iloc[(calls['strike'] - spot).abs().argmin()]
            atm_put  = puts.iloc[(puts['strike']  - spot).abs().argmin()]
            premium = atm_call['bid'] + atm_put['bid']  # sell both at bid
            print(f'  ATM straddle (sell): {atm_call["strike"]:.0f} call @ ${atm_call["bid"]:.2f} + '
                  f'{atm_put["strike"]:.0f} put @ ${atm_put["bid"]:.2f}')
            print(f'  Total premium collected: ${premium:.2f}/share (×100 = ${premium*100:.0f}/contract)')
            print(f'  Break-even range: ${spot-premium:.2f} ← ${spot:.2f} → ${spot+premium:.2f}')
            print(f'  Days to expiry: {chain["days_to_expiry"]}')
            # Position sizing (without going live):
            nav = cfg['starting_capital']
            notional = nav * cfg['capital_at_risk_pct']
            n_contracts = max(1, int(notional / (premium * 100)))
            print(f'  → Recommended size: {n_contracts} contract(s) (~${notional:,.0f} notional)')

            # Future: uncomment to actually place the order
            # _place_order(chain, n_contracts)
    elif sig.decision.startswith('skip'):
        print(f'  No trade today.')

# Run it
daily_run()


## §7 — Broker integration (placeholders)

Each broker has a different API. Below are stubs — fill in `place_straddle`
once you've signed up and gotten an API key. See API_KEYS.md for details.


In [ ]:
def _place_order_tradier(chain, n_contracts, dry_run=True):
    """Tradier example (sandbox). Get key at https://tradier.com/products/brokerage"""
    token = CRED.get('TRADIER_TOKEN')
    if not token:
        print('  No TRADIER_TOKEN in credentials; skipping live order'); return
    url = ('https://api.tradier.com/v1/accounts/{}/orders'.format(CRED.get('TRADIER_ACCOUNT_ID')))
    # Build OCO (one-cancels-other) sell call + sell put
    headers = {'Authorization': f'Bearer {token}', 'Accept': 'application/json'}
    # Multi-leg order construction (combo). See:
    # https://documentation.tradier.com/brokerage-api/trading/place-multileg-order
    print(f'  [STUB] would submit Tradier OCO: {n_contracts} contracts, dry_run={dry_run}')

def _place_order_tastytrade(chain, n_contracts, dry_run=True):
    """TastyTrade. Sign up at https://developer.tastytrade.com/"""
    print('  [STUB] TastyTrade order placeholder')

def _place_order_ibkr(chain, n_contracts, dry_run=True):
    """Interactive Brokers via ib_insync. pip install ib_insync first."""
    print('  [STUB] IBKR order placeholder — requires TWS or IB Gateway running')


## §8 — Operating notes

**When to run:** Once per trading day, after 3:30 PM ET (so VIX and RV are
near-final). Output is "enter today" or "no trade." Positions held 21
trading days. So at any time you'll have 0-2 open positions.

**Position rolling:** When a position hits expiry, it auto-closes. Next
month, signal regenerates. If still `enter`, open new.

**Kill switches** (manual, for now):
- If realized loss in a single trade exceeds -5% of capital: pause for 1 week
- If 3 consecutive trades stop-loss: pause until VIX < 18
- If VIX > 40 mid-trade: close immediately

**What the actual hedging looks like in production:**
- Day 0: sell 1 straddle at strike K, collect premium P
- Day 0: position delta ≈ 0 (calls and puts offset). No SPY trade yet.
- Day 5: SPY rallies, position is now short ~30 delta. Buy 30 shares.
  Wait — short straddle when stock rises means short call grows MORE negative
  delta. So to be delta-neutral, BUY shares. Yes. (Same direction as SPY moved.)
- Day 10: SPY falls below K, position is now long ~30 delta. Sell shares.
- Day 21: expiry, options settle. Cumulative SPY hedge PnL ≈ -RV² × notional.
  Net PnL = premium - hedge PnL ≈ (IV² - RV²) × notional / 2. Matches backtest.

**The "short underlying" part:** happens implicitly through delta-hedge.
When SPY rises, you ADD to your long SPY hedge position (which is "short
exposure" against your short call). When SPY falls hard, you sell SPY (closing
hedge) — that's the "short underlying when option overvalued" mechanic.
